# 06a — 晶体结构、对称性与材料数据库（Pymatgen · CIF · COD · Materials Project）

前五讲我们始终在和**分子**打交道：一个 SMILES 字符串唯一地描述一个有限大小的原子团，`Chem.MolFromSmiles` 读进来就是一张原子-键的图。描述符、指纹、构象、SHAP——整套工具链都建立在“分子是一个有边界的个体”这个前提上。

但如果研究对象是 $\text{SrTiO}_3$、$\text{TiO}_2$ 或者 $\text{CsPbI}_3$ 呢？

- 它们**没有分子**。一块钛酸锶晶体里找不到“一个 $\text{SrTiO}_3$ 分子”，只有无限重复的周期性点阵；
- 化学式 $\text{TiO}_2$ 对应着**至少三种**完全不同的材料：金红石（rutile）、锐钛矿（anatase）、板钛矿（brookite）。它们组成相同，但密度、带隙、光催化活性截然不同——**组成不足以确定材料**；
- 决定性质的是**晶胞参数、空间群对称性、原子的 Wyckoff 位置、配位环境**，这些概念在分子世界里根本不存在。

本节我们换一套工具：用 **pymatgen**（Python Materials Genomics，Materials Project 的官方库）建立晶体结构的表示，做对称性分析、构建超胞与点缺陷，并打通三个真实数据源——本地 **CIF** 文件、开放的 **COD（Crystallography Open Database）** 和 **Materials Project API**。

---

## 0. 环境准备

本节需要 `pymatgen`。如果还没装：

```bash
pip install pymatgen
```

> **⚠️ 关于本讲的运行环境**：`pymatgen` 对 `spglib` / `numpy` / `scipy` 的版本比较敏感，建议在一个干净的虚拟环境或 Colab 里运行本讲。本讲**不需要** Materials Project 的 API Key 也能完整跑完（第 7 节的在线检索是可选的加餐）。

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

warnings.filterwarnings("ignore")

from pymatgen.core import Structure, Lattice, Composition, Element, Species
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.io.cif import CifWriter, CifParser

plt.rcParams["figure.dpi"] = 110
np.random.seed(42)

import pymatgen.core as pmg_core
print("pymatgen 核心模块已载入:", pmg_core.__name__)

def find_file(name):
    for path in (f"data/{name}", f"../data/{name}", name):
        if os.path.exists(path):
            return path
    try:
        from google.colab import files
        print(f"请上传 {name}")
        files.upload()
        return name
    except ImportError:
        raise FileNotFoundError(f"找不到 {name}，请确保它位于 data/ 文件夹或当前目录下")

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---

## 1. 从分子到晶体：周期性边界条件

一个分子由**原子坐标**唯一确定。一个晶体则需要两样东西：

1. **晶格（Lattice）**：三个基矢 $\mathbf{a}, \mathbf{b}, \mathbf{c}$ 张成的平行六面体——**晶胞（unit cell）**。它们的长度 $a, b, c$ 和夹角 $\alpha, \beta, \gamma$ 就是晶胞参数；
2. **基元（Basis）**：晶胞内每个原子的**分数坐标（fractional coordinates）** $(x, y, z)$，取值在 $[0, 1)$。

整个晶体 = 晶胞沿三个基矢方向无限平移复制。任何一个原子的实际笛卡尔坐标是

$$\mathbf{r} = x\,\mathbf{a} + y\,\mathbf{b} + z\,\mathbf{c}$$

**为什么要用分数坐标而不是埃（Å）？** 因为分数坐标把“结构”和“尺寸”解耦了：加压或升温时晶胞会胀缩（$a, b, c$ 改变），但只要对称性不变，分数坐标完全不动。这让不同温度/压力下的同一个相可以直接比较。

**周期性边界条件（PBC）** 是这里最重要的概念：坐标 $(0.9, 0, 0)$ 和 $(-0.1, 0, 0)$ 是**同一个原子**。算两个原子的距离时必须考虑所有周期性镜像，取最近的那个——这叫**最小镜像约定（minimum image convention）**。pymatgen 的 `Structure.get_distance()` 会自动处理。

下面我们手动搭一个最经典的结构：**岩盐型 NaCl**（面心立方，$a = 5.64$ Å）。

In [ ]:
# 方式一：完全手动——给出晶格 + 每个原子的分数坐标
# 岩盐结构的常规立方胞里有 4 个 Na + 4 个 Cl，共 8 个原子
lattice = Lattice.cubic(5.64)

na_sites = [[0.0, 0.0, 0.0], [0.5, 0.5, 0.0], [0.5, 0.0, 0.5], [0.0, 0.5, 0.5]]
cl_sites = [[0.5, 0.5, 0.5], [0.0, 0.0, 0.5], [0.0, 0.5, 0.0], [0.5, 0.0, 0.0]]

nacl_manual = Structure(lattice, ["Na"] * 4 + ["Cl"] * 4, na_sites + cl_sites)

print(nacl_manual)
print()
print(f"化学式           : {nacl_manual.composition.reduced_formula}")
print(f"晶胞内原子数     : {nacl_manual.num_sites}")
print(f"晶胞体积         : {nacl_manual.volume:.2f} A^3")
print(f"理论密度         : {nacl_manual.density:.3f} g/cm^3   (实验值 2.165)")

# 注意：必须让 pymatgen 在周期性边界下找最近镜像，不能随手挑两个编号相减
cl_dists = sorted(nacl_manual.get_distance(0, j) for j in range(4, 8))
print(f"Na(0) 到四个 Cl 的距离: {[round(d, 3) for d in cl_dists]} A")
print(f"最近邻 Na-Cl 距离: {cl_dists[0]:.3f} A   (= a/2 = {5.64 / 2:.3f})")
print(f"次近邻（体对角）  : {cl_dists[-1]:.3f} A   (= sqrt(3)a/2 = {3 ** 0.5 * 5.64 / 2:.3f})")

手动列出 8 个原子已经够烦了。换成有 20 个原子的晶胞呢？幸好**对称性**可以帮我们：岩盐结构属于空间群 $Fm\bar{3}m$（No. 225），只要给出**不等价位置（asymmetric unit）**——Na 在 $(0,0,0)$、Cl 在 $(\frac{1}{2},\frac{1}{2},\frac{1}{2})$——空间群的对称操作会自动把其余原子生成出来。

---

## 2. 用空间群搭建晶体：三个经典结构

`Structure.from_spacegroup(sg, lattice, species, coords)` 正是干这个的。下面用它搭三个贯穿材料科学的结构：

In [ ]:
# ① 岩盐 NaCl：Fm-3m (225)，Na 在 4a，Cl 在 4b
nacl = Structure.from_spacegroup(
    "Fm-3m", Lattice.cubic(5.64), ["Na", "Cl"], [[0, 0, 0], [0.5, 0.5, 0.5]]
)

# ② 立方钙钛矿 SrTiO3：Pm-3m (221)，A 位 Sr 在角，B 位 Ti 在体心，O 在面心
#    这是整个钙钛矿家族（含卤化物钙钛矿太阳能电池）的母结构
srtio3 = Structure.from_spacegroup(
    "Pm-3m", Lattice.cubic(3.905), ["Sr", "Ti", "O"],
    [[0, 0, 0], [0.5, 0.5, 0.5], [0.5, 0.5, 0.0]]
)

# ③ 金刚石结构 Si：Fd-3m (227)，半导体工业的基石
si = Structure.from_spacegroup("Fd-3m", Lattice.cubic(5.431), ["Si"], [[0, 0, 0]])

for name, s in [("NaCl (岩盐)", nacl), ("SrTiO3 (钙钛矿)", srtio3), ("Si (金刚石)", si)]:
    print(f"{name:18s} 化学式={s.composition.reduced_formula:8s} "
          f"原子数={s.num_sites:3d}  V={s.volume:8.2f} A^3  rho={s.density:6.3f} g/cm^3")

print("\n--- SrTiO3 的完整结构 ---")
print(srtio3)

In [ ]:
def plot_structure(struct, ax=None, title=""):
    """把晶胞里的原子画成三维散点，并勾出晶胞的 12 条棱。"""
    if ax is None:
        fig = plt.figure(figsize=(5.2, 4.6))
        ax = fig.add_subplot(111, projection="3d")

    palette = {"Na": "#8c6bb1", "Cl": "#41ab5d", "Sr": "#2c7fb8", "Ti": "#d95f0e",
               "O": "#c0392b", "Si": "#4d4d4d", "Ba": "#2c7fb8", "Pb": "#555555",
               "I": "#6a3d9a", "Cs": "#e7298a", "La": "#1b9e77"}
    sizes = {"O": 90, "Cl": 150, "I": 170}

    coords = np.array([s.coords for s in struct.sites])
    symbols = [s.specie.symbol for s in struct.sites]
    for sym in sorted(set(symbols)):
        m = [i for i, s in enumerate(symbols) if s == sym]
        ax.scatter(coords[m, 0], coords[m, 1], coords[m, 2],
                   s=sizes.get(sym, 190), label=sym, depthshade=True,
                   color=palette.get(sym, "#7f8c8d"), edgecolor="white", linewidth=0.6)

    # 晶胞的 12 条棱：从每个角点沿三个基矢方向各连一条
    mat = struct.lattice.matrix
    for i in (0, 1):
        for j in (0, 1):
            for k in (0, 1):
                origin = i * mat[0] + j * mat[1] + k * mat[2]
                for axis, flag in zip(mat, (i, j, k)):
                    if flag == 0:
                        ax.plot(*zip(origin, origin + axis), color="#999999", lw=0.9)

    ax.set_title(title, fontsize=10)
    ax.set_box_aspect([1, 1, 1])
    ax.set_axis_off()
    ax.legend(fontsize=8, loc="upper left")
    return ax

fig = plt.figure(figsize=(14, 4.4))
for k, (s, t) in enumerate([(nacl, "NaCl  rock salt  Fm-3m"),
                            (srtio3, "SrTiO3  perovskite  Pm-3m"),
                            (si, "Si  diamond  Fd-3m")]):
    ax = fig.add_subplot(1, 3, k + 1, projection="3d")
    plot_structure(s, ax=ax, title=t)
plt.show()

中间那张图就是**钙钛矿 $ABX_3$ 的母结构**：$A$ 位（Sr）占据立方体的八个角，$B$ 位（Ti）在体心，六个 $X$（O）在面心，构成一个包围 $B$ 位的 $\text{BO}_6$ 八面体。把 Sr→Cs、Ti→Pb、O→I，就得到了做太阳能电池的卤化物钙钛矿 $\text{CsPbI}_3$——**同一个几何骨架，完全不同的应用领域**。这也是为什么钙钛矿能成为一个横跨铁电、超导、催化、光伏的巨大材料家族。

### 🏋️ 练习 06a-1：搭建卤化物钙钛矿并比较晶胞

用 `Structure.from_spacegroup` 搭建三个立方钙钛矿，空间群都是 `Pm-3m`，A/B/X 的分数坐标与上面的 $\text{SrTiO}_3$ 完全相同：

| 材料 | $a$ / Å | A | B | X |
|---|---|---|---|---|
| $\text{SrTiO}_3$ | 3.905 | Sr | Ti | O |
| $\text{BaTiO}_3$（立方高温相） | 4.006 | Ba | Ti | O |
| $\text{CsPbI}_3$（$\alpha$ 相） | 6.289 | Cs | Pb | I |

1. 对每个结构打印：化学式、晶胞原子数、体积、密度，以及 **B–X 键长**（先在 `struct.sites` 里找出 B 位和 X 位的编号，再用 `struct.get_distance(i, j)`）；
2. 把三者的 B–X 键长和晶胞参数 $a$ 列成一张表，验证立方钙钛矿里 $a \approx 2\,d_{B-X}$ 这个几何关系；
3. **思考**：$\text{CsPbI}_3$ 的晶胞体积是 $\text{SrTiO}_3$ 的多少倍？

In [ ]:
# 在下方填写你的代码：


---

## 3. 对称性分析：SpacegroupAnalyzer

给定一堆原子坐标，**反推**它的空间群，是晶体学里最基本的操作。pymatgen 通过 `spglib` 实现：`SpacegroupAnalyzer` 在给定容差内寻找所有对称操作（旋转、反演、螺旋轴、滑移面），然后匹配到 230 个空间群之一。

几个必须分清的概念：

| 概念 | 含义 |
|---|---|
| **常规胞（conventional cell）** | 教科书上画的那个、能体现最高对称性的胞（NaCl 的立方胞，8 个原子） |
| **原始胞（primitive cell）** | 体积最小的重复单元（NaCl 的菱面体原始胞，只有 2 个原子） |
| **晶系（crystal system）** | 七大晶系：立方、六方、四方、三方、正交、单斜、三斜 |
| **Wyckoff 位置** | 一组对称等价位置的标记，如 `4a`、`4b`、`8c` |

**为什么要关心原始胞？** 因为 DFT 计算的代价大致按原子数的三次方增长。NaCl 用原始胞算（2 个原子）比用常规胞（8 个原子）快几十倍，而结果完全等价。**第一步永远是化到原始胞**。

In [ ]:
rows = []
for name, s in [("NaCl", nacl), ("SrTiO3", srtio3), ("Si", si)]:
    sga = SpacegroupAnalyzer(s, symprec=0.01)
    prim = sga.get_primitive_standard_structure()
    sym_struct = sga.get_symmetrized_structure()
    rows.append({
        "material":    name,
        "space_group": sga.get_space_group_symbol(),
        "SG_number":   sga.get_space_group_number(),
        "crystal_sys": sga.get_crystal_system(),
        "point_group": sga.get_point_group_symbol(),
        "n_sym_ops":   len(sga.get_symmetry_operations()),
        "conv_sites":  s.num_sites,
        "prim_sites":  prim.num_sites,
        "n_wyckoff":   len(sym_struct.equivalent_sites),
    })

print(pd.DataFrame(rows).to_string(index=False))

print("\n--- SrTiO3 的 Wyckoff 位置分解 ---")
sym_sto = SpacegroupAnalyzer(srtio3).get_symmetrized_structure()
for group, wyck in zip(sym_sto.equivalent_sites, sym_sto.wyckoff_symbols):
    site = group[0]
    print(f"  {site.specie.symbol:3s}  Wyckoff {wyck:4s}  "
          f"多重度 {len(group)}  分数坐标 {np.round(site.frac_coords, 3)}")

读这张表：

- **NaCl** 的 48 个对称操作把 8 个原子压缩成 **2 个不等价位置**（一个 Na、一个 Cl），原始胞只有 2 个原子——DFT 计算量直接降到常规胞的 $(2/8)^3 \approx 1.6\%$；
- **Si** 的常规胞有 8 个原子，原始胞只有 2 个，而且这 2 个原子还是**对称等价**的（同一个 Wyckoff 位置）——这就是为什么硅只有一种键长、一种键角；
- **$\text{SrTiO}_3$** 本身已经是原始胞（5 个原子），三个元素各占一个 Wyckoff 位置，其中 O 的多重度是 3（三个面心位置互相等价）。

> **📌 `symprec` 这个参数很要命**：它是判断“两个位置算不算等价”的坐标容差（默认 0.01，单位是分数坐标）。实验测出来的结构总带一点噪声，`symprec` 太小会把 $Pm\bar{3}m$ 误判成 $P1$（毫无对称性），太大又会把真实的畸变（比如 $\text{BaTiO}_3$ 室温下那个让它具有铁电性的微小四方畸变）抹平。**报告空间群时必须同时报告你用的 `symprec`。**

---

## 4. 超胞与点缺陷：从理想晶体到真实材料

理想晶体在现实中并不存在。真实材料里到处是**点缺陷**——氧空位、间隙原子、取代掺杂——而且往往正是这些缺陷决定了性能：

- $\text{TiO}_2$ 的**氧空位**提供 n 型导电性，是光催化活性的关键；
- $\text{SrTiO}_3$ 掺 Nb 或 La 变成透明导体；
- 卤化物钙钛矿里的**卤素空位**迁移导致 J–V 迟滞。

要在计算里描述一个浓度为 1/8 的缺陷，就得先做一个 $2\times2\times2$ 的**超胞**（把晶胞沿三个方向各复制 2 次），再在超胞里动一个原子。

In [ ]:
# 2x2x2 超胞
supercell = srtio3 * (2, 2, 2)
print(f"原胞 : {srtio3.composition.reduced_formula:8s} {srtio3.num_sites:3d} 个原子")
print(f"超胞 : {supercell.composition.formula:16s} {supercell.num_sites:3d} 个原子  "
      f"V = {supercell.volume:.1f} A^3")

# ① 氧空位：删掉一个 O
vac = supercell.copy()
o_indices = [i for i, s in enumerate(vac.sites) if s.specie.symbol == "O"]
vac.remove_sites([o_indices[0]])
print(f"\n① 氧空位 : {vac.composition.formula}")
print(f"   名义组成 SrTiO(3-delta)，delta = {3 / len(o_indices) * 1:.4f} "
      f"(共 {len(o_indices)} 个 O 位缺 1 个)")

# ② 取代掺杂：把一个 Sr 换成 La（A 位施主掺杂，SrTiO3 常用的 n 型掺杂方式）
doped = supercell.copy()
sr_indices = [i for i, s in enumerate(doped.sites) if s.specie.symbol == "Sr"]
doped.replace(sr_indices[0], "La")
print(f"\n② La 掺杂 : {doped.composition.formula}")
print(f"   掺杂浓度 x = {1 / len(sr_indices):.4f}  (Sr(1-x)La(x)TiO3)")

# ③ 缺陷破坏对称性——这正是我们要的
print()
for label, s in [("理想超胞", supercell), ("含氧空位", vac), ("La 掺杂", doped)]:
    sga = SpacegroupAnalyzer(s, symprec=0.01)
    print(f"{label:10s} 空间群 = {sga.get_space_group_symbol():10s} "
          f"(No. {sga.get_space_group_number():3d})  对称操作数 = {len(sga.get_symmetry_operations())}")

注意最后三行的对称操作数：

- **理想 $2\times2\times2$ 超胞**仍是 $Pm\bar{3}m$，而且操作数从原胞的 48 涨到 384——因为超胞内部多出了 8 倍的平移对称性。**超胞不改变晶体的对称性**，只是换了个更大的重复单元；
- **挖掉一个氧**后，立方对称性直接塌成四方 $P4/mmm$，操作数 384 → 16：空位把三个坐标轴中的一个挑了出来；
- **把一个 Sr 换成 La** 后，点群对称性还是 $m\bar{3}m$（La 坐在角上，三个轴仍然等价），但那 8 倍的平移对称性没了，操作数 384 → 48。

一句话：**缺陷就是对称性破缺**。这对计算是个坏消息（能利用的对称操作变少，计算变慢），但对物理是个好消息——对称性破缺往往正是新性质的来源：铁电极化、Jahn–Teller 畸变、极化子局域化。

### 🏋️ 练习 06a-2：掺杂浓度与超胞大小

对 $\text{SrTiO}_3$：

1. 分别构建 $1\times1\times1$、$2\times2\times2$、$3\times3\times3$、$4\times4\times4$ 超胞，打印每个超胞的原子数、Sr 原子数和体积；
2. 对每个超胞，把**一个** Sr 替换成 La，计算对应的掺杂浓度 $x = 1 / N_{\text{Sr}}$，并打印替换后的化学式；
3. 画一张“超胞倍数 $n$ vs 可达到的最低掺杂浓度 $x$”的图（纵轴用对数坐标）；
4. **思考**：如果实验上要模拟 $x = 0.01$（1 at.% 掺杂）的 $\text{Sr}_{1-x}\text{La}_x\text{TiO}_3$，至少需要多大的超胞？它有多少个原子？按 DFT 计算量随原子数三次方增长估计，它比原胞慢多少倍？

> **提示**：`srtio3 * (n, n, n)` 得到 $n\times n\times n$ 超胞；`Structure.replace(i, "La")` 原地替换第 `i` 个位点。

In [ ]:
# 在下方填写你的代码：


---

## 5. CIF：晶体学的通用语言

**CIF（Crystallographic Information File）** 之于晶体，就相当于 SMILES 之于分子——它是全世界晶体学数据交换的标准格式。一个 CIF 文件里有晶胞参数、空间群、不等价原子位置、测量条件、文献出处。

所有主流数据库（COD、ICSD、CSD、Materials Project）都用 CIF 导出，所有 DFT 软件（VASP、Quantum ESPRESSO）都能读 CIF。

In [ ]:
cif_text = str(CifWriter(srtio3, symprec=0.01))
print(cif_text[:1200])
print("...")

# 写出去再读回来，确认往返无损
with open("srtio3_generated.cif", "w", encoding="utf-8") as f:
    f.write(cif_text)

reloaded = CifParser("srtio3_generated.cif").parse_structures(primitive=False)[0]
print(f"\n读回来的结构: {reloaded.composition.reduced_formula}, "
      f"{reloaded.num_sites} 个原子, "
      f"空间群 {SpacegroupAnalyzer(reloaded).get_space_group_symbol()}")
print(f"与原结构是否匹配: {reloaded.matches(srtio3)}")

---

## 6. COD：完全开放的晶体结构数据库

**COD（Crystallography Open Database，crystallography.net）** 收录了 50 多万条实验测定的晶体结构，**完全开放、无需注册、无需 API Key**——这是它相对于 ICSD（商业数据库）和 CSD（剑桥结构数据库，只收有机与金属有机）的最大优势。

它的 REST 接口很朴素：

```
https://www.crystallography.net/cod/result?formula=O2 Ti&format=json    # 按化学式检索
https://www.crystallography.net/cod/result?text=rutile&format=json       # 按关键词检索
https://www.crystallography.net/cod/<COD-ID>.cif                         # 下载 CIF
```

下面我们用它做一件**具体的实验室工作**：把 $\text{TiO}_2$ 的两个主要多晶型——**锐钛矿（anatase）** 和**金红石（rutile）**——的实验结构抓下来对比。这两个相的光催化活性差别很大（P25 商品催化剂就是二者的混合物），而区分它们正是 XRD 的日常任务（下一节 `06b` 就要做这件事）。

In [ ]:
COD_SEARCH = "https://www.crystallography.net/cod/result"
COD_CIF    = "https://www.crystallography.net/cod/{}.cif"

def cod_search(timeout=40, **params):
    """检索 COD，返回 DataFrame（失败时返回空表而不是抛异常）。"""
    params["format"] = "json"
    try:
        r = requests.get(COD_SEARCH, params=params, timeout=timeout)
        if r.status_code != 200:
            print(f"COD 返回状态码 {r.status_code}")
            return pd.DataFrame()
        return pd.DataFrame(r.json())
    except Exception as exc:
        print("COD 检索失败（可能是网络不通）:", exc)
        return pd.DataFrame()

# COD 的化学式写法：元素按字母序、空格分隔、省略下标 1
hits = cod_search(formula="O2 Ti")
if len(hits):
    print(f"COD 中 TiO2 的结构条目共 {len(hits)} 条\n")
    cols = [c for c in ["file", "formula", "mineral", "sg", "a", "c", "year"] if c in hits.columns]
    named = hits[hits["mineral"].notna()] if "mineral" in hits.columns else hits
    print("其中标注了矿物名的前 12 条：")
    print(named[cols].head(12).to_string(index=False))
    if "mineral" in hits.columns:
        print("\n矿物名分布：")
        print(hits["mineral"].value_counts().head(5).to_string())
else:
    print("未获取到 COD 结果，下面的小节会自动改用内置结构。")

In [ ]:
# 两个有代表性的 COD 条目：锐钛矿与金红石
COD_IDS = {"anatase": 7206075, "rutile": 9015662}

# 离线回退：用空间群 + 文献晶胞参数直接搭出来
FALLBACK = {
    "anatase": dict(sg="I4_1/amd", lat=Lattice.tetragonal(3.785, 9.514),
                    sp=["Ti", "O"], co=[[0, 0, 0], [0, 0, 0.2081]]),
    "rutile":  dict(sg="P4_2/mnm", lat=Lattice.tetragonal(4.594, 2.959),
                    sp=["Ti", "O"], co=[[0, 0, 0], [0.3053, 0.3053, 0]]),
}

tio2 = {}
for name, cod_id in COD_IDS.items():
    struct, source = None, ""
    try:
        txt = requests.get(COD_CIF.format(cod_id), timeout=40).text
        struct = CifParser.from_str(txt).parse_structures(primitive=False)[0]
        source = f"COD #{cod_id}"
    except Exception as exc:
        print(f"{name}: 在线下载失败（{exc}），改用内置晶胞参数")
    if struct is None:
        fb = FALLBACK[name]
        struct = Structure.from_spacegroup(fb["sg"], fb["lat"], fb["sp"], fb["co"])
        source = "内置文献参数"
    tio2[name] = struct

    sga = SpacegroupAnalyzer(struct, symprec=0.01)
    lat = struct.lattice
    print(f"{name:8s} [{source:12s}]  {struct.composition.reduced_formula:6s}  "
          f"SG={sga.get_space_group_symbol():10s}(No.{sga.get_space_group_number():3d})  "
          f"a={lat.a:.3f}  c={lat.c:.3f} A   rho={struct.density:.3f} g/cm3")

d_ana, d_rut = tio2["anatase"].density, tio2["rutile"].density
print(f"\n密度差 = {(d_rut - d_ana) / d_ana:.1%}（金红石更致密）——"
      f"这正是工业上可以用沉降法把两相分开的原因。")

In [ ]:
# 局部配位环境对比：两个相里 Ti 周围都是 6 个 O 构成的八面体，但畸变方式不同
for name, s in tio2.items():
    ti_idx = next(i for i, site in enumerate(s.sites) if site.specie.symbol == "Ti")
    dists = sorted(nn.nn_distance for nn in s.get_neighbors(s[ti_idx], r=2.5))
    print(f"{name:8s} Ti 的近邻 O 个数 = {len(dists)}")
    print(f"{'':8s} Ti-O 键长 = {', '.join(f'{d:.3f}' for d in dists)} A")
    print(f"{'':8s} 平均 {np.mean(dists):.3f} A，畸变度 (max-min)/mean = "
          f"{(max(dists) - min(dists)) / np.mean(dists):.1%}\n")

fig = plt.figure(figsize=(10, 4.4))
for k, (name, s) in enumerate(tio2.items()):
    ax = fig.add_subplot(1, 2, k + 1, projection="3d")
    plot_structure(s, ax=ax, title=f"TiO2 - {name}")
plt.show()

两个相的 $\text{TiO}_6$ 八面体都由 1 个 Ti 和 6 个 O 组成，键长也都在 1.95 Å 附近——**局部环境几乎一样**。它们的区别完全在于**八面体之间怎么连接**：金红石是共棱八面体沿 $c$ 轴排成直链，锐钛矿则以之字形连接。就是这个拓扑差异，让锐钛矿的带隙（约 3.2 eV）比金红石（约 3.0 eV）宽，密度低约 10%，光催化活性却高得多。

**这正是晶体信息学和分子信息学最根本的区别**：分子的性质主要由“有哪些原子、怎么连”决定；晶体的性质还要加上“这些单元在空间里怎么堆”。

---

## 7. Materials Project API（可选加餐）

**Materials Project（MP，materialsproject.org）** 是目前规模最大的计算材料数据库：15 万以上的无机化合物，每一条都有 DFT 算出的形成能、带隙、弹性常数、磁矩、能带结构、态密度。它和 COD 的分工是：

| | COD | Materials Project |
|---|---|---|
| 数据来源 | **实验**测定的晶体结构 | **DFT 计算**的结构与性质 |
| 性质数据 | 基本没有 | 形成能、带隙、弹性、磁性…… |
| 访问方式 | 完全开放 | 免费，但需注册拿 API Key |
| 典型用途 | 查“这个相实验上存在吗” | 高通量筛选、相图、稳定性判据 |

用 MP 需要先在 materialsproject.org 注册并在个人页面拿一个 API Key，然后：

```bash
pip install mp-api
```

```python
from mp_api.client import MPRester

with MPRester("你的_API_KEY") as mpr:
    # 按化学体系批量检索
    docs = mpr.materials.summary.search(
        chemsys="Ti-O",
        fields=["material_id", "formula_pretty", "symmetry",
                "formation_energy_per_atom", "energy_above_hull", "band_gap"],
    )

    # 按性质窗口筛选（高通量筛选的入口）
    candidates = mpr.materials.summary.search(
        band_gap=(1.1, 1.7),            # 光伏吸收层的最佳带隙窗口
        energy_above_hull=(0, 0.05),    # 亚稳范围内
        num_elements=(2, 3),
        fields=["material_id", "formula_pretty", "band_gap", "energy_above_hull"],
    )

    structure = mpr.get_structure_by_material_id("mp-2657")   # 金红石 TiO2
```

> **⚠️ 不要把 API Key 写进 Notebook 再提交到 GitHub。** 正确做法是放进环境变量：
> ```python
> import os
> api_key = os.environ["MP_API_KEY"]      # 缺失时直接 KeyError，而不是静默出错
> ```
> 这条规则对所有需要密钥的服务都成立。

**本讲剩下的内容完全不依赖 MP。** 下一节 `06b` 要用的形成能、带隙、$E_{\text{hull}}$ 数据，已经以 CSV 的形式打包在 `data/` 里（来自 AFLOW 与已发表的钙钛矿数据集），所以没有 API Key 也能完整跑完。

下面这一格会检测环境变量里有没有 Key：有就真的去查一次，没有就跳过。

In [ ]:
mp_key = os.environ.get("MP_API_KEY", "")

if not mp_key:
    print("未检测到环境变量 MP_API_KEY，跳过在线检索。")
    print("想试的话：注册 materialsproject.org -> 拿 API Key -> 设置环境变量 MP_API_KEY -> 重跑本格。")
else:
    try:
        from mp_api.client import MPRester
        with MPRester(mp_key) as mpr:
            docs = mpr.materials.summary.search(
                chemsys="Ti-O",
                fields=["material_id", "formula_pretty", "symmetry",
                        "formation_energy_per_atom", "energy_above_hull", "band_gap"],
            )
        mp_df = pd.DataFrame([{
            "mp_id":          d.material_id,
            "formula":        d.formula_pretty,
            "spacegroup":     d.symmetry.symbol,
            "Ef_eV_atom":     d.formation_energy_per_atom,
            "E_hull_eV_atom": d.energy_above_hull,
            "band_gap_eV":    d.band_gap,
        } for d in docs]).sort_values("E_hull_eV_atom")
        print(f"Materials Project 中的 Ti-O 体系共 {len(mp_df)} 条：")
        print(mp_df.head(15).round(4).to_string(index=False))
    except ImportError:
        print("未安装 mp-api，请先 pip install mp-api")
    except Exception as exc:
        print("MP 检索失败:", exc)

### 🏋️ 练习 06a-3：用 COD 做一次真实的物相调研

假设你在实验上合成了一个 $\text{BaTiO}_3$ 样品，需要搞清楚它在文献里一共有哪些已知的多晶型。

1. 用本节的 `cod_search()` 按化学式检索 $\text{BaTiO}_3$（注意 COD 的化学式格式：元素按**字母序**、空格分隔、省略下标 1，所以写成 `"Ba O3 Ti"`）；
2. 统计返回结果里 `sg`（空间群）列的取值分布，找出出现次数最多的 3 个空间群；
3. 从其中两个主要空间群里各挑一条记录，下载 CIF、用 `CifParser` 解析，打印化学式、晶胞参数 $a, b, c$、密度，以及 `SpacegroupAnalyzer` 重新判定的空间群；
4. **讨论**：$\text{BaTiO}_3$ 室温下是四方相 $P4mm$（具有铁电性），高温下是立方相 $Pm\bar{3}m$（顺电）。对比你找到的四方相与立方相，计算四方畸变量 $c/a - 1$。这个数值只有百分之一量级。现在把 `SpacegroupAnalyzer` 的 `symprec` 从 `0.01` 一路扫到 `0.5`（建议取 `0.01, 0.05, 0.1, 0.2, 0.3, 0.5`），记录每个取值下判定出的空间群和对称操作数，找出四方相被“抹平”成立方相的**临界 symprec**。结合你算出的 $c/a - 1$ 和 Ti/O 的位移量，解释这个临界值为什么落在那个量级。

> **提示**：`hits["sg"].value_counts()` 可以直接统计空间群分布。如果课堂网络不通，用内置参数自己搭：立方相 $a = 4.006$ Å（`Pm-3m`，坐标同正文的 $\text{SrTiO}_3$）；四方相 $a = 3.994$ Å、$c = 4.038$ Å（`P4mm`，Ti 位 $(0.5, 0.5, 0.515)$，O 位 $(0.5, 0.5, -0.025)$ 与 $(0.5, 0, 0.485)$）。

In [ ]:
# 在下方填写你的代码：


---

## 小结

| 核心概念 | pymatgen 实现 | 材料科学意义 |
|---|---|---|
| **晶格 + 基元** | `Lattice.cubic(a)` + `Structure(lattice, species, coords)` | 分数坐标把“结构”与“尺寸”解耦；周期性边界条件下 $(0.9,0,0)$ 与 $(-0.1,0,0)$ 是同一个原子 |
| **空间群建模** | `Structure.from_spacegroup(sg, lat, species, coords)` | 只需给出不等价位置，230 个空间群的对称操作自动生成全部原子 |
| **对称性分析** | `SpacegroupAnalyzer(s, symprec=0.01)` | 原始胞 vs 常规胞决定 DFT 计算量（NaCl: 2 vs 8 个原子）；`symprec` 必须随空间群一起报告 |
| **超胞与点缺陷** | `struct * (2,2,2)`、`remove_sites`、`replace` | 缺陷浓度由超胞大小决定；**缺陷即对称性破缺**，是铁电、极化子、掺杂导电的根源 |
| **CIF 往返** | `CifWriter` / `CifParser` | 晶体学的通用交换格式，等价于分子世界的 SMILES |
| **COD 数据库** | `requests` + REST，无需 Key | 50 万条**实验**结构；锐钛矿 vs 金红石：局部 $\text{TiO}_6$ 八面体几乎相同，差别全在八面体的连接拓扑 |
| **Materials Project** | `mp-api` + `MPRester`（需 Key） | 15 万条 **DFT** 计算性质，高通量筛选的主力数据源；密钥走环境变量，绝不写进 Notebook |

**下一节预告（`06b_xrd_phase_diagram_screening.ipynb`）**：有了结构，就能算出它的**模拟 XRD 谱图**——这是把计算结构和实验衍射图谱对上号的桥梁，也是判断“我合成出来的到底是锐钛矿还是金红石”的标准手段。再往前一步，我们要回答材料信息学最核心的一个问题：**这个化合物到底能不能稳定存在？** 答案藏在热力学凸包（convex hull）里。最后，我们会把化学式变成特征向量、训练带隙预测模型，走通一条完整的高通量虚拟筛选漏斗——并看到一个触目惊心的现象：随机划分时 $R^2 = 0.83$ 的模型，一旦要求它外推到没见过的化学体系，$R^2$ 会掉到 $0.19$。